In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"
CHUNKED_LAWS_INDEX_PATH = INDEX_PATH / "chunked_laws_index.pkl"
CHUNKED_COURTS_INDEX_PATH = INDEX_PATH / "chunked_courts_index.pkl"


# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.28 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
# law_court_reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
law_court_reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

print("加载成功")

加载成功


In [4]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [5]:
import dense_index
from dense_index import DenseIndex
from sparse_index import SparseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648


In [6]:
law_sparse_index = SparseIndex(dense_model, "../data/processed/_dense_sparse_law", law_doc)
law_sparse_index.load()

In [7]:
import json
citation_idf_d = {}
with open("../data/citation_idf.jsonl") as inf:
    for line in inf:
        d = json.loads(line.strip())
        citation_idf_d[d['citation']] = d['idf']

In [8]:
import citation_utils
import rerank_utils

VALID_RECALL_PKL = "../data/processed/valid_recall.pkl"
law_topk=1000
court_topk=1000

all_hits_l = []
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

for id, query, gold_citations in tqdm(zip(valid_df['query_id'].tolist(), 
                                      valid_df['query2'].tolist(), 
                                      valid_df['gold_citations'].tolist()), 
                                  total=len(valid_df), 
                                  desc="valid-data") :

    court_recall = court_dense_index.search_with_score(query, top_k=court_topk)
    law_recall = law_sparse_index.search_with_score(query, top_k=law_topk)

    reranked_court = rerank_utils.rerank_by_dense_batch_chunked(law_court_reranker, query, [hit for hit,score in court_recall], len(court_recall), 10, 384, 128)

    top_k = 40
    while top_k < len(reranked_court):
        court_first_layer = [(hit['citation'], score) for hit, score in reranked_court[:top_k]]
        second_layer = citation_utils.second_layer_citation_with_score(court_consideration_d, law_d, court_first_layer)

        all_hits = []
        all_hits.extend([doc for doc, score in reranked_court[:5]])

        with_idf_second_layer = []
        
        for citation,score in second_layer:
            if citation in law_d:
                all_hits.append({'citation':citation, 'text':law_d[citation], 'type':'L'})
            
        if len(all_hits) >= 45:
            break

        top_k += 40

    all_hits_l.append(all_hits)
    
    print("second_layer.len:", len(second_layer), 'first_layer.len:', len(court_first_layer), "all_hits.len:", len(all_hits))
    
print(len(all_hits_l), len(all_hits_l[0]))

valid-data:   0%|          | 0/10 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
valid-data:  10%|█         | 1/10 [00:37<05:34, 37.20s/it]

second_layer.len: 112 first_layer.len: 320 all_hits.len: 48


valid-data:  20%|██        | 2/10 [01:28<06:03, 45.41s/it]

second_layer.len: 218 first_layer.len: 320 all_hits.len: 48


valid-data:  30%|███       | 3/10 [02:18<05:32, 47.52s/it]

...

valid-data:  90%|█████████ | 9/10 [08:46<01:02, 62.77s/it]

second_layer.len: 170 first_layer.len: 80 all_hits.len: 56


valid-data: 100%|██████████| 10/10 [10:14<00:00, 61.43s/it]

second_layer.len: 183 first_layer.len: 80 all_hits.len: 53
10 48



  0%|          | 0/10 [00:00<?, ?it/s]


In [9]:
def cal_recall2(all_hits_l, gold_citations_l, limit=50):
    recalls = []
    for all_hits, gold_citations in zip(all_hits_l, gold_citations_l):
        
        all_citation = []
        for hit in all_hits[:50]:
            all_citation.append(hit['citation'])

        hits = len(set(all_citation) & set(gold_citations))

        # print("gold_citations.len:", len(gold_citations), ", hits.len:", hits)
        recall = hits / len(gold_citations)
        recalls.append(recall)
        
    mean_recall = np.mean(recalls)
    return mean_recall

In [10]:
def cal_precision2(all_hits_l, gold_citations_l, limit=50):
    precisions = []
    for all_hits, gold_citations in zip(all_hits_l, gold_citations_l):
        
        all_citation = []
        for hit in all_hits[:limit]:
            all_citation.append(hit['citation'])
        
        predicted = set(all_citation)
        hits = len(predicted & set(gold_citations))
        
        if len(predicted) == 0:
            precision = 0.0
        else:
            precision = hits / len(predicted)
        
        precisions.append(precision)
        
    mean_precision = np.mean(precisions)
    return mean_precision

In [14]:
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

r = cal_recall2(all_hits_l, valid_df['gold_citations'].apply(lambda x: x.split(";")), 30)
p = cal_precision2(all_hits_l, valid_df['gold_citations'].apply(lambda x: x.split(";")), 30)
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("r:",r, "p:",p, "f1:",f1)

r: 0.19050657162704893 p: 0.11333333333333336 f1: 0.1421192176005591


In [12]:
rerank_court_law_hits_l_l = []
from tqdm import tqdm
import rerank_utils
import reranker

# r = reranker.Reranker("/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3", "../ft_data/lora_reranker_output_30000_e1_0-145")

for idx, (_, row) in tqdm(enumerate(valid_df.iterrows()), total=len(valid_df)):
    query = row['query2']
    
    # court_pairs = []
    # for hit,score in courts_hits_l[idx]:
    #     court_pairs.append((query, hit['text']))
    # ret = r.compute_score(court_pairs)
    # l2 = []
    # for i, score in ret[:5]:
    #     l2.append(courts_hits_l[idx][i])
    # rerank_courts_hits_l.append(l2)
    # # rerank_courts_hits_l.append([])

    
    rerank_court_law_l = rerank_utils.rerank_by_dense_batch_chunked(law_court_reranker, query, all_hits_l[idx], 40, 10, 384, 128)

    law_count = 0
    court_count = 0

    for item, score in rerank_court_law_l:
        if item['type'] == 'C':
            court_count += 1
        elif item['type'] == 'L':
            law_count += 1
    print("all_hits.len:", len(all_hits_l[idx]), "law_count:", law_count, "court_count:", court_count)

    rerank_court_law_hits_l_l.append(rerank_court_law_l)

r = cal_recall2(rerank_court_law_hits_l_l, valid_df['gold_citations'].apply(lambda x: x.split(";")))
p = cal_precision2(rerank_court_law_hits_l_l, valid_df['gold_citations'].apply(lambda x: x.split(";")))
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
print("r:",r, "p:",p, "f1:",f1)

KeyError: 'type'